# 0. Environment setting

## Libary import

In [1]:
import pandas as pd
from datetime import date, timedelta
import re
import requests
import json
import urllib3
from difflib import get_close_matches
import ctypes
import threading
import time
import os
from urllib.parse import quote
requests.packages.urllib3.disable_warnings()
from io import StringIO

## Qualtrics Credentials

In [2]:
# ============================================================
# CONFIGURATION 
# ============================================================

API_TOKEN = "dZZueEgbaPrCuSTXEtIp2sz0EqlJNxg93jYEod7U"   # <-- insert your token
DATA_CENTER = "iad1"
SURVEYS_ID = {
    "CMP" : "SV_6zfVwcNb8KrpiPI",
    "RGE" : "SV_4HFHCtt06kaQv0a",
    "NSE" : "SV_eyAxdOqDXvF3qu2"
}

## iQor SFTP Credentials

In [3]:
"""
BASE_URL = 'https://mft.iqor.com'
USERNAME = 'Iberdrola'
PASSWORD = 'KaC#9eta'
"""

"\nBASE_URL = 'https://mft.iqor.com'\nUSERNAME = 'Iberdrola'\nPASSWORD = 'KaC#9eta'\n"

In [4]:
BASE_URL = 'https://mft.iqor.com'
USERNAME = 'Avangrid_Automation'
PASSWORD = 'K3P#9e$td'

## Success/failure message
- Note: This just just work on a windows environment this need to be removed and replaced with an email/team notification

In [5]:
def popup(message, title="Info", timeout=10, is_error=False):
    MB_OK = 0x0
    icon = 0x10 if is_error else 0x40  # ERROR vs INFORMATION
    
    def show_box():
        ctypes.windll.user32.MessageBoxW(0, message, title, MB_OK | icon)
    
    t = threading.Thread(target=show_box)
    t.start()
    t.join(timeout=timeout)
    
    hwnd = ctypes.windll.user32.FindWindowW(None, title)
    if hwnd:
        ctypes.windll.user32.PostMessageW(hwnd, 0x0010, 0, 0)

# Keep your original names as simple wrappers if you want
def popup_info(message, title="Success", timeout=10):
    popup(message, title, timeout, is_error=False)

def popup_error(message, title="Error", timeout=10):
    popup(message, title, timeout, is_error=True)

## Get sharepoint landing folder

In [6]:
def get_onedrive_path():
    return os.path.join(os.path.expanduser("~"), "OneDrive - IBERDROLA S.A")

def get_sharepoint_folder(opco):
    """
    Returns the landing folder path for a given OpCo.
    opco: 'CMP', 'RGE', or 'NSE'
    """
    OPCO_FOLDERS = {
        "CMP": "iQor_CMP",
        "RGE": "iQor_RGE",
        "NSE": "iQor_NYSEG"
    }
    
    if opco not in OPCO_FOLDERS:
        raise ValueError(f"Unknown OpCo: {opco}. Must be one of {list(OPCO_FOLDERS.keys())}")
    
    onedrive_root = get_onedrive_path()
    return os.path.join(
        onedrive_root,
        "iQor-Avangrid - General",
        OPCO_FOLDERS[opco]
    )

def get_output_path(filename, opco):
    """
    Returns (full_path, folder_exists) for a given filename and OpCo.
    """
    sharepoint_folder = get_sharepoint_folder(opco)
    if os.path.exists(sharepoint_folder):
        return os.path.join(sharepoint_folder, filename), True
    return filename, False

# 1. Extract

## Initial data extraction
Task:
- Get all theraw data from the file despite the format
- Get the column names from the file context (If the order changes the dataframe keeps it consistent)

In [7]:
"""session = requests.Session()
session.verify = False
session.auth = (USERNAME, PASSWORD)
session.get(BASE_URL + '/files', timeout=15)"""

"session = requests.Session()\nsession.verify = False\nsession.auth = (USERNAME, PASSWORD)\nsession.get(BASE_URL + '/files', timeout=15)"

In [8]:
# ============================================================
# EXTRACT
# ============================================================

session = requests.Session()
session.verify = False
session.auth = (USERNAME, PASSWORD)

# Warmup - allow server to fully establish session
session.get(BASE_URL + '/files', timeout=15)
time.sleep(2)
session.get(BASE_URL + '/Report/Survey%20Reports/', timeout=15)
time.sleep(2)

SFTP_FOLDERS = {
    "CMP": "/Report/Survey%20Reports/CMP/",
    "RGE": "/Report/Survey%20Reports/RGE/",
    "NSE": "/Report/Survey%20Reports/NSE/",
}

def list_dir(path):
    r = session.get(BASE_URL + path, timeout=15)
    files, folders = [], []
    for line in r.text.strip().splitlines()[1:]:
        parts = line.split()
        if len(parts) < 9:
            continue
        perms, _, owner, group, size, month, day, time_str, *name_parts = parts
        name = ' '.join(name_parts)
        if name in ('.', '..'):
            continue
        is_dir = perms.startswith('d')
        entry = {
            'name' : name,
            'size' : int(size),
            'date' : f'{month} {day} {time_str}',
            'type' : 'DIR' if is_dir else 'FILE'
        }
        (folders if is_dir else files).append(entry)
    return files, folders

def get_latest_file(opco):
    folder_path  = SFTP_FOLDERS[opco]
    files, _     = list_dir(folder_path)
    time.sleep(1)
    reports      = [
        f for f in files
        if 'Daily Survey Report_' in f['name']
        and 'Triage' not in f['name']
    ]
    if not reports:
        raise FileNotFoundError(f"No Daily Survey Reports found for {opco}")
    reports.sort(key=lambda f: f['name'].split('_')[-1].replace('.csv', ''))
    latest       = reports[-1]
    print(f"[{opco}] Latest file : {latest['name']}")
    print(f"[{opco}] Server date : {latest['date']}")
    encoded_name = quote(latest['name'])
    r            = session.get(BASE_URL + folder_path + encoded_name, timeout=30)
    r.raise_for_status()
    parts        = latest['name'].replace('.csv', '').split('_')
    company      = parts[0].split()[0]
    # ⚠️ TYPE THESE TWO LINES BY HAND IN JUPYTER
    report_date  = pd.to_datetime(parts[-1], format='%Y%m%d')
    df           = pd.read_csv(StringIO(r.text))
    # ⚠️ END
    #df['company']     = company
    #df['report_date'] = report_date
    print(f"[{opco}] Shape       : {df.shape}")
    return df, latest['name'], r.text

# --- Run ---
dataframes = {}
raw_files  = {}
for opco in SFTP_FOLDERS:
    try:
        df, filename, raw_text = get_latest_file(opco)
        dataframes[opco]       = df
        raw_files[opco]        = (filename, raw_text)
    except Exception as e:
        print(f"[{opco}] ERROR: {e}")

df_cmp = dataframes.get("CMP")
df_rge = dataframes.get("RGE")
df_nse = dataframes.get("NSE")

[CMP] Latest file : CMP Daily Survey Report_20260615.csv
[CMP] Server date : Jun 16 07:00:24
[CMP] Shape       : (83, 15)
[RGE] Latest file : RGE Daily Survey Report_20260615.csv
[RGE] Server date : Jun 16 07:00:21
[RGE] Shape       : (234, 14)
[NSE] Latest file : NSE Daily Survey Report_20260615.csv
[NSE] Server date : Jun 16 07:00:12
[NSE] Shape       : (412, 14)


In [9]:
# ============================================================
# LOCAL ONLY — Save raw CSVs to SharePoint for audit/pivot use
# Remove this entire cell when deploying to Azure
# ============================================================

for opco, (filename, raw_text) in raw_files.items():
    try:
        local_path, folder_exists = get_output_path(filename, opco)
        if folder_exists:
            with open(local_path, 'w', encoding='utf-8', newline='') as f:
                f.write(raw_text)
            print(f"[{opco}] ✓ Saved to : {local_path}")
        else:
            print(f"[{opco}] ⚠ Skipped  : Folder not found — {local_path}")
    except PermissionError:
        print(f"[{opco}] ⚠ Skipped  : File is open in Excel or locked — {filename}")
    except Exception as e:
        print(f"[{opco}] ⚠ Skipped  : {e}")

print("\nLocal save complete — continuing pipeline...")

[CMP] ✓ Saved to : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_CMP\CMP Daily Survey Report_20260615.csv
[RGE] ✓ Saved to : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_RGE\RGE Daily Survey Report_20260615.csv
[NSE] ✓ Saved to : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_NYSEG\NSE Daily Survey Report_20260615.csv

Local save complete — continuing pipeline...


In [10]:
display(df_cmp.head(3))
display(df_nse.head(3))
display(df_rge.head(3))
print("hello world")

,ID,Name,Date_Time,Work_Group,InteractionID,Phone_Number,Survey_Name,CSAT1,NPS,I_C,C_K,FCR,Call_Reason,Survey_Status_Count,Survey_Status
0,38999530,kevin.dallum,06/15/2026 07:43:25,CMP.USUT.CS.RESCRCL,708964313976,2074584660,CMP IQR Survey w/ NPS,5.0,5.0,#,#,#,3.0,6.0,COMPLETED
1,69353693,adreana.cepeda,06/15/2026 07:57:06,CMP.USUT.CS.RESCRCL,708964314841,2073448931,CMP IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED
2,38999525,cachae.perry,06/15/2026 08:04:24,CMP.USUT.CS.RESCRCL,708964318903,2075083074,CMP IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED


,ID,Name,Date,WorkGroup,InteractionID,Telephone,SurveyName,NPS,FCR,CSAT,E_H,C_E,CallReason,SurveyStatus
0,39116708,ashley.johnson17,06/15/2026 07:10:34,NSE.USUT.FE.NYCRCL,708964304643,7164491922,NSE IQR Survey w/ NPS,10.0,1.0,5.0,5.0,5.0,3.0,COMPLETED
1,38869873,keshia.clay,06/15/2026 07:11:55,NSE.USUT.FE.NYCRCL,708964303971,2015654381,NSE IQR Survey w/ NPS,9.0,1.0,5.0,5.0,5.0,3.0,COMPLETED
2,69353685,reiceljane.torres,06/15/2026 07:13:42,NSE.USUT.FE.GEN,708964305809,8452946851,NSE IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED


,ID,Name,Date,WorkGroup,InteractionID,Telephone,SurveyName,NPS,FCR,CSAT,E_H,C_E,CallReason,SurveyStatus
0,44080957,lavanda.alexander,06/15/2026 07:06:56,RGE.USUT.FE.RGCRCL,708964303410,5855765408,RGE IQR Survey w/ NPS,7.0,1.0,4.0,4.0,5.0,6.0,COMPLETED
1,38869946,tameka.robinson2,06/15/2026 07:10:46,RGE.USUT.FE.GEN,708964304777,5852844695,RGE IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED
2,60920122,jessalucille.bendaa,06/15/2026 07:14:45,RGE.USUT.FE.GEN,708964304638,5852037651,RGE IQR Survey w/ NPS,9.0,1.0,5.0,5.0,5.0,6.0,COMPLETED


hello world


In [11]:
print("CMP:",df_cmp.columns.tolist())
print("NSE:",df_nse.columns.tolist())
print("RGE:",df_rge.columns.tolist())

CMP: ['ID', 'Name', 'Date_Time', 'Work_Group', 'InteractionID', 'Phone_Number', 'Survey_Name', 'CSAT1', 'NPS', 'I_C', 'C_K', 'FCR', 'Call_Reason', 'Survey_Status_Count', 'Survey_Status']
NSE: ['ID', 'Name', 'Date', 'WorkGroup', 'InteractionID', 'Telephone', 'SurveyName', 'NPS', 'FCR', 'CSAT', 'E_H', 'C_E', 'CallReason', 'SurveyStatus']
RGE: ['ID', 'Name', 'Date', 'WorkGroup', 'InteractionID', 'Telephone', 'SurveyName', 'NPS', 'FCR', 'CSAT', 'E_H', 'C_E', 'CallReason', 'SurveyStatus']


# 2. Transform

## Add metadata columns (Survey Status/Completion & Tag)
Tasks:
- Delete empty columns
- Compute Survey Completion and Tag Fields
- Reorder the fields matching the Qualtrics survey order

In [12]:
# ============================================================
# TRANSFORM
# ============================================================

# --- Column rename maps per OpCo ---
RENAME_MAP = {
    "CMP": {
        "Date_Time": "Date/Time",
        "CSAT1"    : "CSAT",
        "Survey_Status" : "Survey Completion",
    },
    "RGE": {
        "Date"     : "Date/Time",
        "Telephone" : "Phone Number",
    },
    "NSE": {
        "Date"     : "Date/Time",
        "Telephone" : "Phone Number",
    },
}

# --- Score columns to cast to integer per OpCo ---
SCORE_COLS = {
    "CMP": ["NPS", "CSAT", "Call_Reason", "Survey_Status_Count"],
    "RGE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
    "NSE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
}

def transform(df, opco):
    df = df.copy()

    # Step 1 — Rename columns
    df = df.rename(columns=RENAME_MAP.get(opco, {}))

    # Step 2 — Fix Date/Time format
    if "Date/Time" in df.columns:
        df["Date/Time"] = pd.to_datetime(df["Date/Time"], errors="coerce")
        df["Date/Time"] = df["Date/Time"].dt.strftime("%m/%d/%Y %H:%M:%S")

    # Step 3 — Cast score columns to integer
    for col in SCORE_COLS[opco]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    print(f"[{opco}] Transform complete | Shape: {df.shape}")
    return df

# --- Run ---
df_cmp = transform(df_cmp, "CMP")
df_rge = transform(df_rge, "RGE")
df_nse = transform(df_nse, "NSE")

[CMP] Transform complete | Shape: (83, 15)
[RGE] Transform complete | Shape: (234, 14)
[NSE] Transform complete | Shape: (412, 14)


In [13]:
# DEBUG - run after transform, before prepare
print("C_K in df_cmp.columns:", "C_K" in df_cmp.columns)
print("C_K all null:", df_cmp["C_K"].isna().all())
print("C_K dtype:", df_cmp["C_K"].dtype)
print("C_K sample:\n", df_cmp["C_K"].value_counts(dropna=False))

C_K in df_cmp.columns: True
C_K all null: False
C_K dtype: object
C_K sample:
 C_K
NaN    45
#      36
0       2
Name: count, dtype: int64


In [14]:
# ============================================================
# PREPARE FOR QUALTRICS
# ============================================================

# Expected columns per OpCo for validation
EXPECTED_COLS = {
    "CMP": ["ID", "Name", "Date/Time", "Work_Group", "InteractionID", "Phone_Number",
            "Survey_Name", "CSAT", "NPS", "I_C", "C_K", "FCR", "Call_Reason",
            "Survey_Status_Count", "Survey Completion", "company", "report_date","Tag"],
    "RGE": ["ID", "Name", "Date/Time", "WorkGroup", "InteractionID", "Phone Number",
            "SurveyName", "NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason",
            "SurveyStatus", "company", "report_date","Tag"],
    "NSE": ["ID", "Name", "Date/Time", "WorkGroup", "InteractionID", "Phone Number",
            "SurveyName", "NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason",
            "SurveyStatus", "company", "report_date","Tag"],
}

# Required scoring columns per OpCo
REQUIRED_SCORE_COLS = {
    "CMP": ["CSAT", "NPS", "I_C", "C_K", "FCR", "Call_Reason"],
    "RGE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
    "NSE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
}

# Work Group column name per OpCo
WORKGROUP_COL = {
    "CMP": "Work_Group",
    "RGE": "WorkGroup",
    "NSE": "WorkGroup",
}

# Survey Status column name per OpCo
SURVEY_STATUS_COL = {
    "CMP": "Survey_Status",
    "RGE": "SurveyStatus",
    "NSE": "SurveyStatus",
}

def prepare_for_qualtrics(df, opco):
    df = df.copy()

    # Step 1 — Drop empty columns
    #df = df.dropna(axis=1, how="all")

    # Step 2 — Warn on unknown columns
    unknown_cols = [c for c in df.columns if c not in EXPECTED_COLS[opco]]
    if unknown_cols:
        popup_error(
            f"[{opco}] Unknown columns detected:\n{unknown_cols}",
            title="ETL Warning"
        )
        print(f"[{opco}] ⚠ Unknown columns: {unknown_cols}")

    # Step 3 — Validate required scoring columns
    required = REQUIRED_SCORE_COLS[opco]
    missing  = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"[{opco}] Missing required scoring columns: {missing}")

    # Step 1 — Drop empty columns
    df = df.dropna(axis=1, how="all")
    
    # Step 4 — Forward fill ID and Name
    df[["ID", "Name"]] = df[["ID", "Name"]].ffill()

    # Step 5 — Tag test rows (case insensitive)
    wg_col    = WORKGROUP_COL[opco]
    df["Tag"] = df[wg_col].str.contains("test", case=False, na=False).map({
        True : "Test",
        False: ""
    })

    print(f"[{opco}] Prepare complete | Shape: {df.shape}")
    if df["Tag"].eq("Test").any():
        print(f"[{opco}] ⚠ Test rows found: {df['Tag'].eq('Test').sum()}")

    return df

# --- Run ---
df_cmp = prepare_for_qualtrics(df_cmp, "CMP")
df_rge = prepare_for_qualtrics(df_rge, "RGE")
df_nse = prepare_for_qualtrics(df_nse, "NSE")

[CMP] Prepare complete | Shape: (83, 16)
[RGE] Prepare complete | Shape: (234, 15)
[NSE] Prepare complete | Shape: (412, 15)


## Prepare format for Qualtrics
Tasks:
- Get real field names form the qualtrics targeted survey
- Map dataframe field with official survey field names
- Add extra metadata rows for qualtrics understanding

In [15]:
def prepare_for_qualtrics(df, opco):
    survey_id = SURVEYS_ID[opco]
    
    url     = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{survey_id}"
    headers = {"X-API-TOKEN": API_TOKEN}
    
    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()
    
    survey_json   = response.json()
    label_map     = survey_json["result"]["questions"]
    
    qid_to_name   = {qid: q["questionName"] for qid, q in label_map.items()}
    qid_to_text   = {qid: q["questionText"] for qid, q in label_map.items()}
    name_to_qid   = {name: qid for qid, name in qid_to_name.items()}
    qualtrics_names = list(name_to_qid.keys())

    # Step 1 — Fuzzy match df columns to Qualtrics questionName
    mapped_cols = {}
    used_names  = set()
    for col in df.columns:
        match = get_close_matches(col, qualtrics_names, n=1, cutoff=0.6)
        if match:
            new_name = match[0]
            if new_name in used_names:
                new_name = col
            mapped_cols[col] = new_name
            used_names.add(new_name)
        else:
            mapped_cols[col] = col

    df = df.rename(columns=mapped_cols)

    # Step 2 — Build questionText row
    row2 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        row2.append(qid_to_text.get(qid, "") if qid else "")

    # Step 3 — Build ImportId row
    row3 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        row3.append(f'{{"ImportId": "{qid}_TEXT"}}' if qid else "")

    # Step 4 — Stack headers + data
    hdr1     = pd.DataFrame([list(df.columns)], columns=df.columns)
    hdr2     = pd.DataFrame([row2],             columns=df.columns)
    hdr3     = pd.DataFrame([row3],             columns=df.columns)
    df_final = pd.concat([hdr1, hdr2, hdr3, df.reset_index(drop=True)], ignore_index=True)

    # Step 5 — Remove duplicate header row
    df_final = df_final.iloc[1:].reset_index(drop=True)

    print(f"[{opco}] Qualtrics prep complete | Shape: {df_final.shape}")
    display(df_final.head(5))
    
    return df_final

# --- Run ---
df_cmp_q = prepare_for_qualtrics(df_cmp, "CMP")
df_rge_q = prepare_for_qualtrics(df_rge, "RGE")
df_nse_q = prepare_for_qualtrics(df_nse, "NSE")

[CMP] Qualtrics prep complete | Shape: (85, 16)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,CSAT,NPS,I_C,C_K,FCR,Call_Reason,Survey Status Count,Survey Completion,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,CSAT,NPS,I_C,C_K,FCR,Call_Reason,Survey Status Count,Survey Completion,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID15_TEXT""}","{""ImportId"": ""QID16_TEXT""}","{""ImportId"": ""QID14_TEXT""}"
2,38999530,kevin.dallum,06/15/2026 07:43:25,CMP.USUT.CS.RESCRCL,708964313976,2074584660,CMP IQR Survey w/ NPS,5,5,#,#,#,3,6,COMPLETED,
3,69353693,adreana.cepeda,06/15/2026 07:57:06,CMP.USUT.CS.RESCRCL,708964314841,2073448931,CMP IQR Survey w/ NPS,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,ABANDONED,
4,38999525,cachae.perry,06/15/2026 08:04:24,CMP.USUT.CS.RESCRCL,708964318903,2075083074,CMP IQR Survey w/ NPS,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,ABANDONED,


[RGE] Qualtrics prep complete | Shape: (236, 15)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,E_H,C_E,Call Reason,Survey Status,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,Ease of Help,Clear Explanation,Call Reason,Survey Status,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID14_TEXT""}","{""ImportId"": ""QID15_TEXT""}"
2,44080957,lavanda.alexander,06/15/2026 07:06:56,RGE.USUT.FE.RGCRCL,708964303410,5855765408,RGE IQR Survey w/ NPS,7,1,4,4,5,6,COMPLETED,
3,38869946,tameka.robinson2,06/15/2026 07:10:46,RGE.USUT.FE.GEN,708964304777,5852844695,RGE IQR Survey w/ NPS,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABANDONED,
4,60920122,jessalucille.bendaa,06/15/2026 07:14:45,RGE.USUT.FE.GEN,708964304638,5852037651,RGE IQR Survey w/ NPS,9,1,5,5,5,6,COMPLETED,


[NSE] Qualtrics prep complete | Shape: (414, 15)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,E_H,C_E,Call Reason,Survey Status,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,Ease of Help,Clear Explanation,Call Reason,Survey Status,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID14_TEXT""}","{""ImportId"": ""QID15_TEXT""}"
2,39116708,ashley.johnson17,06/15/2026 07:10:34,NSE.USUT.FE.NYCRCL,708964304643,7164491922,NSE IQR Survey w/ NPS,10,1,5,5,5,3,COMPLETED,
3,38869873,keshia.clay,06/15/2026 07:11:55,NSE.USUT.FE.NYCRCL,708964303971,2015654381,NSE IQR Survey w/ NPS,9,1,5,5,5,3,COMPLETED,
4,69353685,reiceljane.torres,06/15/2026 07:13:42,NSE.USUT.FE.GEN,708964305809,8452946851,NSE IQR Survey w/ NPS,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABANDONED,


# 3. Load

## File csv creation after transformation
Tasks:
- Create repository file
- Create Queatrics ready temporary file

In [ ]:
def load_data(df, output_path):
    df.to_csv(output_path, index=False, encoding="utf-8")

    return


## Source folder mapping

In [ ]:
# Extract Date fields
today = date.today()

# If today is Monday (weekday() == 0), use last Saturday
if today.weekday() == 0:
    effective_date = today - timedelta(days=2)
else:
    effective_date = today

# Extract Date fields
source_year = effective_date.year
source_month = effective_date.strftime("%B")
source_day = effective_date.strftime("%d")
source_weekday = effective_date.weekday()
landing_yesterday = effective_date - timedelta(days=1)

#print(source_year, source_month, source_day, source_weekday, landing_yesterday)


## Post to Qualtrics

In [ ]:
def upload_to_qualtrics(file_path, DATA_CENTER, SURVEY_ID, API_TOKEN):
    """
    Uploads a CSV file to Qualtrics using the Import Responses API.
    Expects a fully formatted Qualtrics-ready CSV at file_path.
    """

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}/import-responses"

    headers = {
        "X-API-TOKEN": API_TOKEN,
        "Content-Type": "text/csv",
        "charset": "UTF-8"
    }

    print(f"\nUploading file to Qualtrics: {file_path}")

    with open(file_path, "rb") as f:
        response = requests.post(
            url,
            headers=headers,
            data=f,
            verify=False
        )

    print("\n=== RAW RESPONSE TEXT ===")
    print(response.text)
    print("=========================\n")

    try:
        result = response.json()
        print("Upload response (parsed):")
        print(json.dumps(result, indent=4))
        return result
    except Exception:
        print("Could not parse JSON response.")
        return response.text


# Execute

In [ ]:
try :
    input_file_path = rf"\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\NY_post_call_survey\daily\{source_year}\{source_month}\{source_day}\NY Feedback Daily.xls"
    #input_file_path = r"\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\NY_post_call_survey\daily\2026\April\03\NY Feedback Daily.xls"
    #output_file_path = r"~\Desktop\NY Feedback Daily Cleaned March 5.csv"
    #output_file_path = "NY_qualtrics_upload6.csv"   # local temp file
    #output_file_path = f"NY Feedback Daily {landing_yesterday}.csv"
    filename = f"NY Feedback Daily {landing_yesterday}.csv"
    output_file_path, used_sharepoint = get_output_path(filename)

    temp_file_path = "NY_qualtrics_upload.csv"   # local temp file
    data = extract_data(input_file_path)
    cleaned_data = transform_data(data)
    load_data(cleaned_data, output_file_path)
    ready_data = prepare_for_qualtrics(cleaned_data)
    load_data(ready_data, temp_file_path)
    result = upload_to_qualtrics(temp_file_path, DATA_CENTER, SURVEY_ID, API_TOKEN)
    
except Exception as e:
    error_message = f"Error running the program:\n{e}"

    if today.weekday() == 0:
        error_message += "\n\nMonday run — Friday folder expected."
    elif today.weekday() == 6:
        error_message += "\n\nWeekend run — folder may be empty."

    popup_error(error_message)

else:
    popup_info(filename,"Upload successful")
